# Audio-Text Alignment Validation

Additional validation of the alignment between the audio features and the (frozen)
text features, using a trained **fusion** checkpoint. Run this notebook from the
repository root.

Two experiments, both comparing **matched** audio-text pairs against **unmatched**
pairs (the text features `t` shuffled within each batch, so every audio clip is
paired with some other utterance's text):

1. **Feature alignment** — MMD between `feat_audio` and `t` inside `AudioEnc`
   (`src/model__gan.py`), computed over the whole evaluation split (overall and
   per class), matched vs shuffled, plus two pair-sensitive per-sample
   statistics (see the *Metrics* section for why they are needed).
2. **Fusion recognition** — emotion recognition through the fused head
   (`logits_fuse`), reporting WA / UA / UAR for matched vs shuffled pairs. If the
   fusion head really uses the text modality, accuracy should drop for shuffled pairs.

Requirements: the Stage-2c audio checkpoint (`checkpoints/audio_fusion.pt`) and the
Stage-1 text encoder (`checkpoints/text.pt`); the 7B text backbone is loaded, so a
GPU with enough memory is needed.

In [1]:
import os
import warnings
from contextlib import nullcontext
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import Wav2Vec2Config, Wav2Vec2Processor

import common
from measure_WA_UA import unweighted_accuracy, uar_score, weighted_accuracy
from src.datacollator import DataCollatorCTCWithPadding
from src.dataset import Dataset_aug

warnings.filterwarnings("ignore")

/home2/csahn/emotion-aae-clean/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Arguments

Edit this cell and run everything below. The defaults mirror
`evaluate.py --stage fusion`.

In [2]:
from types import SimpleNamespace

# Which GPU(s) to use. Must be set before the first CUDA call in this
# process (i.e. before the torch.cuda.is_available() below) to take effect.
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

args = SimpleNamespace(
    # data
    dataset="msp-podcast",       # "iemocap" | "msp-improv" | "msp-podcast"
    data_root="../MSP-PODCAST-Publish-1.10/",              # None -> common.DEFAULT_DATA_ROOT[dataset]
    split="test",                # "test" | "dev"
    fold=0,                      # IEMOCAP / MSP-IMPROV only
    refresh_cache=False,         # re-parse instead of loading cache/
    # models
    ckpt="../F_Pllama_bal_gan_augmented_0.pt",   # Stage-2c fusion checkpoint to analyse
    text_ckpt="../F_P_llama20k_0.pt",      # Stage-1 (frozen) text encoder
    text_backbone=common.DEFAULT_TEXT_BACKBONE,
    audio_backbone=common.DEFAULT_AUDIO_BACKBONE,
    # run
    batch_size=8,
    max_batches=None,            # int -> stop after N batches (quick smoke run)
    seed=230,                    # seeds the within-batch shuffle
    csv_out="alignment_results.csv",  # results row is appended here (see last cell)
)

if args.data_root is None:
    args.data_root = common.DEFAULT_DATA_ROOT[args.dataset]
assert os.path.exists(args.ckpt), f"ckpt '{args.ckpt}' not found"
assert os.path.exists(args.text_ckpt), f"text_ckpt '{args.text_ckpt}' not found"

torch.manual_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Data

In [3]:
dataset = common.build_dataset(args.dataset, args.data_root, args.refresh_cache)
num_labels = dataset.num_labels
_, dev_examples, test_examples = common.resolve_splits(dataset, args.fold)
examples = test_examples if args.split == "test" else dev_examples

id2label = {}
for name, idx in dataset.label_list.items():
    id2label.setdefault(idx, name)
class_names = [id2label.get(i, str(i)) for i in range(num_labels)]
print(f"{args.dataset} {args.split}: {len(examples)} examples, classes = {class_names}")

loaded dataset cache: cache/msp-podcast_min5.pkl
msp-podcast test: 10779 examples, classes = ['N', 'H', 'A', 'S']


## Models

The fusion audio encoder plus the frozen Stage-1 text encoder (this loads the 7B
backbone).

In [4]:
config_a = Wav2Vec2Config.from_pretrained(args.audio_backbone, num_labels=num_labels)
config_a.proj_size = config_a.hidden_size
processor = Wav2Vec2Processor.from_pretrained(args.audio_backbone)

model = common.build_audio_encoder(config_a, num_labels, args.audio_backbone, ckpt=args.ckpt)
model.to(device)
model.eval()

txt, _, tokenizer = common.build_text_encoder(
    args.text_backbone, num_labels, device, freeze=True, ckpt=args.text_ckpt)
txt.eval()

eval_set = Dataset_aug(list(examples))
eval_set.set_tokenizer(tokenizer)
loader = DataLoader(eval_set, batch_size=args.batch_size, shuffle=False,
                    collate_fn=DataCollatorCTCWithPadding(processor=processor, padding=True),
                    num_workers=0)

loaded audio encoder from checkpoint: ../F_Pllama_bal_gan_augmented_0.pt
loaded text encoder from checkpoint: ../F_P_llama20k_0.pt


## Metrics

All statistics are computed over the **whole evaluation split**: the run loop only
accumulates the per-sample features across batches, and the metrics are evaluated
on the concatenated tensors. This avoids two problems with per-batch statistics:
the empirical MMD estimator is strongly biased at batch size 8 (a mean of
per-batch MMDs is not the whole-set MMD), and a mean of per-batch means weights
the smaller final batch incorrectly while its std understates the per-sample
spread.

**A caveat about MMD and shuffling.** `MMD(x, y)` compares the two *sets* of
vectors — every cross term `k(x_i, y_j)` enters the sum — so it is invariant to
reordering `y`. Shuffling `t` within batches permutes the set but does not change
it, so the **overall** matched and shuffled MMD are identical *by construction*
(up to floating-point rounding). The overall value is still reported — as an
absolute audio–text distribution distance and as a sanity check — but the
discriminative MMD test is **per class**: for the audio features of class *c* the
matched text features are the class-*c* texts, while the shuffled ones are an
approximately class-marginal mixture, so real class-conditional alignment shows
up as per-class MMD(matched) < MMD(shuffled). (For the majority class the two
mixtures are similar, so its gap is expected to be small.)

Two **pair-sensitive** per-sample companions measure the instance-level alignment
the shuffle is meant to break, using the same RBF bandwidth mixture as
`src/MMD.py`:

- **paired kernel distance** — `k(x_i, x_i) + k(y_i, y_i) - 2 k(x_i, y_i)` per pair
  (the MMD integrand restricted to the paired diagonal). Lower = closer pairs.
- **paired cosine similarity** — `cos(feat_audio_i, t_i)` per pair. Higher = closer pairs.

If audio–text alignment is instance-level (not just distribution-level), both
should clearly separate matched from shuffled pairs.

Note: a random permutation leaves each item in place with probability `1/B`, so a
small fraction of "shuffled" pairs is still matched; this only shrinks the gaps
slightly.


In [5]:
BANDWIDTHS = (10, 15, 20, 50)  # same RBF mixture as src/MMD.py

def rbf_paired(x, y):
    """Row-wise kernel k(x_i, y_i) with the MMD bandwidth mixture."""
    d = ((x - y) ** 2).sum(dim=1)
    return sum(torch.exp(-0.5 * d / a) for a in BANDWIDTHS)

def paired_kernel_distance(x, y):
    """Per-pair kernel distance (one value per sample); sensitive to pairing."""
    return rbf_paired(x, x) + rbf_paired(y, y) - 2 * rbf_paired(x, y)

def mmd_full(x, y, device, block=4096):
    """Whole-set MMD with the same kernel and (biased) estimator as src/MMD.py,
    accumulated blockwise so the full NxN kernel matrices never materialise."""
    def mean_k(a, b):
        total = 0.0
        for i in range(0, a.shape[0], block):
            ai = a[i:i + block].to(device)
            for j in range(0, b.shape[0], block):
                bj = b[j:j + block].to(device)
                d = torch.cdist(ai, bj).pow_(2)
                total += sum(torch.exp(-0.5 * d / bw).sum().item() for bw in BANDWIDTHS)
        return total / (a.shape[0] * b.shape[0])
    return mean_k(x, x) + mean_k(y, y) - 2 * mean_k(x, y)


## Run

One forward pass per batch: the matched pass goes through `AudioEnc.forward`
(giving `feat_audio` as `rep_real` and the matched `logits_fuse`); the shuffled
fusion logits reuse the same `feat_audio` through the fused head
`model.hj(cat(feat_audio, t[perm]))`, exactly the `logits_fuse` computation in
`AudioEnc.forward`.

The per-sample features (`feat_audio`, matched `t`, shuffled `t`) are stored on
the CPU in float32 (~60 MB each per 15k examples); all statistics are computed on
the concatenated tensors in the cells below.


In [6]:
amp = (torch.autocast(device.type, dtype=torch.float16)
       if device.type == "cuda" else nullcontext())
gen = torch.Generator().manual_seed(args.seed)

feats, t_matched, t_shuffled = [], [], []
preds_matched, preds_shuffled, targets = [], [], []

with torch.no_grad():
    for b, data in enumerate(tqdm(loader, unit="batch", dynamic_ncols=True)):
        if args.max_batches is not None and b >= args.max_batches:
            break
        perm = torch.randperm(data["labels"].shape[0], generator=gen).to(device)
        with amp:
            t, _ = txt(data, device)            # (B, 1024) text features
            out = model(data, device, t=t)      # matched fusion forward
            logits_shuf = model.hj(torch.cat((out.rep_real, t[perm]), dim=1))

        # accumulate per-sample features; all statistics are computed afterwards
        feats.append(out.rep_real.float().cpu())
        t_matched.append(t.float().cpu())
        t_shuffled.append(t[perm].float().cpu())

        # experiment 2 -- fusion emotion recognition
        preds_matched.extend(torch.argmax(out.logits_fuse, dim=-1).cpu().tolist())
        preds_shuffled.extend(torch.argmax(logits_shuf, dim=-1).cpu().tolist())
        targets.extend(torch.argmax(data["labels"], dim=-1).view(-1).cpu().tolist())

feats = torch.cat(feats)          # (N, 1024) audio features
t_matched = torch.cat(t_matched)  # (N, 1024) matched text features
t_shuffled = torch.cat(t_shuffled)
y = np.array(targets)
print(f"done: {len(y)} examples")


100%|██████████| 1348/1348 [05:49<00:00,  3.86batch/s]


done: 10779 examples


## Experiment 1 — feature alignment, matched vs shuffled

Whole-split statistics. The overall MMD is expected to be identical in both
columns (same set of text features — see *Metrics*); the per-class MMD and the
paired per-sample statistics carry the actual matched-vs-unmatched signal.


In [7]:
pkd_m = paired_kernel_distance(feats, t_matched).numpy()
pkd_s = paired_kernel_distance(feats, t_shuffled).numpy()
cos_m = F.cosine_similarity(feats, t_matched).numpy()
cos_s = F.cosine_similarity(feats, t_shuffled).numpy()

print(f"whole-split statistics over {len(y)} examples\n")
print(f"{'MMD (rbf), overall':<26}  matched = {mmd_full(feats, t_matched, device):.6f}   "
      f"shuffled = {mmd_full(feats, t_shuffled, device):.6f}   (identical by construction)")

def summarize(name, m, s):
    print(f"{name:<26}  matched = {m.mean():.6f} (+/- {m.std():.6f})   "
          f"shuffled = {s.mean():.6f} (+/- {s.std():.6f})")

summarize("paired kernel distance", pkd_m, pkd_s)
summarize("paired cosine similarity", cos_m, cos_s)

print("\nPer-class, matched / shuffled (class of the audio side):")
name_w = max(len(c) for c in class_names)
print(f"  {'class':<{name_w}}  {'MMD':^19}  {'paired kernel dist':^19}  "
      f"{'paired cosine':^19}      n")
for i, cname in enumerate(class_names):
    mask = y == i
    if not mask.any():
        continue
    idx = torch.from_numpy(np.flatnonzero(mask))
    mmd_c_m = mmd_full(feats[idx], t_matched[idx], device)
    mmd_c_s = mmd_full(feats[idx], t_shuffled[idx], device)
    print(f"  {cname:<{name_w}}  {mmd_c_m:8.4f} / {mmd_c_s:8.4f}  "
          f"{pkd_m[mask].mean():8.4f} / {pkd_s[mask].mean():8.4f}  "
          f"{cos_m[mask].mean():8.4f} / {cos_s[mask].mean():8.4f}  {mask.sum():5d}")


whole-split statistics over 10779 examples



MMD (rbf), overall          matched = 1.618744   shuffled = 1.618744   (identical by construction)
paired kernel distance      matched = 7.981972 (+/- 0.016354)   shuffled = 7.984876 (+/- 0.014615)
paired cosine similarity    matched = 0.159762 (+/- 0.094488)   shuffled = 0.033609 (+/- 0.143196)

Per-class, matched / shuffled (class of the audio side):
  class          MMD          paired kernel dist      paired cosine         n
  N    1.6844 /   1.6775    7.9815 /   7.9846    0.1549 /   0.0323   5469
  H    1.7104 /   1.7155    7.9827 /   7.9853    0.1629 /   0.0347   3970
  A    1.8706 /   1.8699    7.9825 /   7.9854    0.1700 /   0.0311    803
  S    1.7246 /   1.7155    7.9805 /   7.9841    0.1704 /   0.0428    537


### Cosine-similarity table — whole split and per class

Paired cosine similarity between each audio feature and its matched / shuffled
text feature: whole split first, then per emotion class (class of the audio
side). `gap` = matched − shuffled; instance-level alignment should make it
clearly positive in every class.

In [8]:
masks = [("overall", np.ones_like(y, dtype=bool))]
masks += [(c, y == i) for i, c in enumerate(class_names) if (y == i).any()]

def _fmt(v):
    return f"{v.mean():.4f} ± {v.std():.4f}"

cos_table = pd.DataFrame(
    [{"class": name,
      "matched": _fmt(cos_m[m]), "shuffled": _fmt(cos_s[m]),
      "gap": f"{cos_m[m].mean() - cos_s[m].mean():.4f}",
      "n": int(m.sum())}
     for name, m in masks]).set_index("class")
cos_table

,matched,shuffled,gap,n
class,,,,
overall,0.1598 ± 0.0945,0.0336 ± 0.1432,0.1262,10779
N,0.1549 ± 0.0932,0.0323 ± 0.1410,0.1227,5469
H,0.1629 ± 0.0945,0.0347 ± 0.1439,0.1282,3970
A,0.1700 ± 0.1059,0.0311 ± 0.1560,0.1389,803
S,0.1704 ± 0.0859,0.0428 ± 0.1401,0.1277,537


## Experiment 2 — fusion recognition, matched vs shuffled

Same audio, same labels; only the text side of the fused head is swapped. The gap
(especially in UA / UAR) quantifies how much the fusion head relies on the correct,
aligned text features.

In [9]:
y = np.array(targets).reshape(-1)
p_m = np.array(preds_matched)
p_s = np.array(preds_shuffled)

for name, p in [("matched", p_m), ("shuffled", p_s)]:
    print(f"{name:>9}:  WA = {weighted_accuracy(list_y_true=y, list_y_pred=p):.4f}   "
          f"UA = {unweighted_accuracy(list_y_true=y, list_y_pred=p):.4f}   "
          f"UAR = {uar_score(labels=y, predictions=p):.4f}")

print(f"\nprediction changed by the text swap on {(p_m != p_s).mean():.1%} of examples")

print("\nPer-class recall (matched / shuffled):")
name_w = max(len(c) for c in class_names)
for i, cname in enumerate(class_names):
    mask = y == i
    rm = (p_m[mask] == i).mean() if mask.any() else float("nan")
    rs = (p_s[mask] == i).mean() if mask.any() else float("nan")
    print(f"  {cname:<{name_w}}  {rm:.4f} / {rs:.4f}   (n={mask.sum()})")

  matched:  WA = 0.5320   UA = 0.6090   UAR = 0.6090
 shuffled:  WA = 0.5181   UA = 0.5781   UAR = 0.5781

prediction changed by the text swap on 22.1% of examples

Per-class recall (matched / shuffled):
  N  0.4270 / 0.4176   (n=5469)
  H  0.6186 / 0.6113   (n=3970)
  A  0.7385 / 0.6949   (n=803)
  S  0.6518 / 0.5885   (n=537)


## Append this run's results to a CSV

One row per notebook run (e.g. one per fusion checkpoint), appended to
`args.csv_out` — the header is written only when the file does not exist yet, so
running the notebook once per checkpoint accumulates a table. Re-running with the
same checkpoint appends a duplicate row (the timestamp tells them apart; keep the
last one). Use one CSV per dataset, since the per-class columns depend on the
class names.

This cell is self-contained after the *Run* cell: it recomputes its statistics
from the stored features, so the experiment cells above do not need to have run.


In [10]:
p_m = np.array(preds_matched)
p_s = np.array(preds_shuffled)
pkd_m = paired_kernel_distance(feats, t_matched).numpy()
pkd_s = paired_kernel_distance(feats, t_shuffled).numpy()
cos_m = F.cosine_similarity(feats, t_matched).numpy()
cos_s = F.cosine_similarity(feats, t_shuffled).numpy()

row = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "ckpt": args.ckpt,
    "split": args.split,
    "n_examples": len(y),
    # the overall MMD is identical for matched/shuffled (set statistic)
    "mmd_overall": mmd_full(feats, t_matched, device),
    "pkd_matched": pkd_m.mean(), "pkd_shuffled": pkd_s.mean(),
    "cos_matched": cos_m.mean(), "cos_shuffled": cos_s.mean(),
    "wa_matched": weighted_accuracy(list_y_true=y, list_y_pred=p_m),
    "wa_shuffled": weighted_accuracy(list_y_true=y, list_y_pred=p_s),
    "ua_matched": unweighted_accuracy(list_y_true=y, list_y_pred=p_m),
    "ua_shuffled": unweighted_accuracy(list_y_true=y, list_y_pred=p_s),
    "uar_matched": uar_score(labels=y, predictions=p_m),
    "uar_shuffled": uar_score(labels=y, predictions=p_s),
    "flip_rate": float((p_m != p_s).mean()),
}
for i, cname in enumerate(class_names):
    mask = y == i
    if not mask.any():
        continue
    idx = torch.from_numpy(np.flatnonzero(mask))
    row[f"recall_{cname}_matched"] = float((p_m[mask] == i).mean())
    row[f"recall_{cname}_shuffled"] = float((p_s[mask] == i).mean())
    row[f"mmd_{cname}_matched"] = mmd_full(feats[idx], t_matched[idx], device)
    row[f"mmd_{cname}_shuffled"] = mmd_full(feats[idx], t_shuffled[idx], device)
    row[f"cos_{cname}_matched"] = float(cos_m[mask].mean())
    row[f"cos_{cname}_shuffled"] = float(cos_s[mask].mean())

# rewrite the file instead of appending so rows written before a column
# was added (e.g. the per-class cosine) stay aligned with the header
new = pd.DataFrame([row])
if os.path.exists(args.csv_out):
    new = pd.concat([pd.read_csv(args.csv_out), new], ignore_index=True)
new.to_csv(args.csv_out, index=False)
print(f"appended results for '{args.ckpt}' to {args.csv_out}")


appended results for '../F_Pllama_bal_gan_augmented_0.pt' to alignment_results.csv
